In [7]:
import NN_model_helper
import numpy as np, pandas as pd, optuna, torch
from sklearn.model_selection import StratifiedKFold
from NN_model import ImprovedNN 
from pathlib import Path
import json, torch
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
from NN_model_helper import (evaluate_fold, plot_training_progress, find_optimal_clusters)

import sys
from pathlib import Path

/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# classifier/ → MELTING_POINT_2026/
PROJECT_ROOT = Path.cwd().parent        # directory above a path: .../MELTING_POINT_2026

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data_curation" / "processed_data"

data_path = processed_dir / "final_dataset.parquet"
df = pd.read_parquet(data_path)

print("Loaded:", data_path)
print("Shape:", df.shape)

Loaded: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/data_curation/processed_data/final_dataset.parquet
Shape: (17220, 122)


In [9]:
df.head()

,SMILES,MP,Type,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
0,ON=Cc1cscc1,122.0,Train,1,0.0,0.0,1,32.985231,0,0,...,1,0.0,168.564167,0,0.000000,0,0.410848,0,0.749989,11.421854
1,O=C1CC[C@]2(C(=C1)CC[C@@H]1[C@@H]2[C@H](O)C[C@...,205.5,Train,1,0.0,0.0,1,0.000000,1,0,...,0,0.0,3553.423052,0,77.105125,0,0.392773,0,8.850244,34.482001
2,[O-][n+]1ccccc1,64.0,Train,1,0.0,0.0,0,0.000000,0,0,...,1,0.0,138.007504,0,0.000000,0,0.618694,0,0.000000,0.000000
3,OC1CCC2(C(C1)CC=C1C2CCC2(C1CCC2C(CCC(C(C)C)C)C...,146.0,Train,1,0.0,0.0,1,0.000000,1,0,...,0,0.0,2479.213445,0,111.854606,0,0.393120,0,55.138771,11.210494
4,CC(=O)c1ccc(cc1)Br,51.0,Train,1,0.0,0.0,0,158.762615,0,0,...,0,0.0,237.939433,0,6.923737,0,0.294512,0,1.250748,5.783245


In [10]:
df.describe()

,MP,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,RDKit_FpDensityMorgan1 X RDKit_NumRotatableBonds,RDKit_NHOHCount X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
count,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,...,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000
mean,126.040571,0.978688,3.025448,0.976507,1.097909,138.151883,0.290012,0.006620,2.838261,0.085017,...,0.593670,2.691309,1215.041434,0.084959,19.166147,0.240244,0.394746,0.035075,10.562985,18.358919
std,70.914541,0.144428,4.629141,5.091234,1.305635,224.123470,0.824084,0.088625,2.475850,0.561846,...,0.491162,10.297792,1639.353590,0.313165,27.956990,0.768413,0.113005,0.298360,22.831053,16.826626
min,0.000000,0.000000,0.000000,0.000000,0.000000,-942.806447,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-0.061902,0.000000,0.000000,0.000000,0.000000,-0.061902
25%,69.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.181818,0.000000,...,0.000000,0.000000,270.794610,0.000000,0.000000,0.000000,0.325709,0.000000,1.258586,7.109798
50%,120.000000,1.000000,0.000000,0.000000,1.000000,69.508109,0.000000,0.000000,2.400000,0.000000,...,1.000000,0.000000,628.851074,0.000000,6.923737,0.000000,0.417985,0.000000,4.346087,14.703796
75%,175.000000,1.000000,4.794537,0.000000,2.000000,174.859973,0.000000,0.000000,4.090909,0.000000,...,1.000000,0.000000,1606.391404,0.000000,25.683286,0.000000,0.481173,0.000000,11.397890,23.662430
max,492.500000,1.000000,65.856226,133.522836,25.000000,6374.741017,11.000000,4.000000,20.460000,36.000000,...,1.000000,266.155326,65644.728169,4.000000,335.074616,25.000000,1.000000,25.000000,694.930816,224.669012


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler

# 2) Filter to TRAIN only
df_test = df[df["Type"].astype(str).str.lower() == "test"].copy()

# 3) Drop non-feature columns
NON_FEATURES = ["SMILES", "MP", "Type", "Ro5"]
feature_cols = [c for c in df_test.columns if c not in NON_FEATURES]

# (prevents issues if any non-numeric columns sneak in)
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_test[c])]

X_test = df_test[feature_cols].to_numpy(dtype=np.float32)

BASE = Path.cwd()
scaler_path = BASE / "artifacts" / "df_final_scaler.pkl"
scaler = joblib.load(scaler_path)

X_test_scaled = scaler.transform(X_test)

# 7) Rebuild DataFrame with SAME column order
df_test_scaled = df_test.copy()
df_test_scaled[feature_cols] = X_test_scaled

# 8) Save as Parquet
out_path = BASE / "artifacts" / "df_test_scaled.parquet"
df_test_scaled.to_parquet(out_path, index=False)

print("Test rows:", df_test_scaled.shape[0])
print("Num features:", len(feature_cols))
print("Saved scaled TEST dataset to:", out_path)

Test rows: 5166
Num features: 118
Saved scaled TEST dataset to: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/artifacts/df_test_scaled.parquet


In [12]:
df_test_scaled.head()

,SMILES,MP,Type,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
12054,[O-][N+](=O)c1c(C)c(C(=O)C)c(c(c1C(C)(C)C)[N+]...,135.5,Test,1,-0.651134,-0.195586,-0.839721,-0.618301,-0.350082,-0.077211,...,0.821926,-0.258763,-0.385976,-0.268728,0.998128,-0.316404,-0.871431,-0.145223,-0.020079,-0.165679
12055,CN(NC(=O)CCC(=O)O)C,154.5,Test,1,0.432729,-0.195586,0.694148,-0.618301,-0.350082,-0.077211,...,0.821926,-0.258763,-0.775069,-0.268728,-0.226208,-0.316404,0.764344,-0.145223,-0.315950,1.029142
12056,CCCCc1ccc(cc1)NC(=O)Oc1ccc(cc1)OC,143.0,Test,1,0.386341,-0.195586,-0.072787,0.145154,-0.350082,-0.077211,...,0.821926,-0.258763,-0.027957,-0.268728,0.252697,1.006958,0.900505,-0.145223,0.410717,-0.307419
12057,OC(=O)COCCN1C(=O)c2c(C1=O)cccc2,128.0,Test,1,0.386341,-0.195586,-0.072787,-0.103564,0.862879,-0.077211,...,0.821926,-0.258763,-0.173134,-0.268728,-0.687065,1.006958,0.749574,-0.145223,0.097925,1.698546
12058,CCCCCCCCCCCCCCC,10.0,Test,1,-0.651134,-0.195586,-0.839721,-0.618301,-0.350082,-0.077211,...,-1.216654,-0.258763,-0.775069,-0.268728,2.805458,-0.316404,-2.878269,-0.145223,1.258066,-1.078546


In [13]:
df_test_scaled.describe()

,MP,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,RDKit_FpDensityMorgan1 X RDKit_NumRotatableBonds,RDKit_NHOHCount X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
count,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,...,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000,5166.000000
mean,125.129235,0.978707,0.011777,-0.023013,0.007677,-0.029054,0.005635,0.009359,-0.011412,-0.000114,...,-0.021366,-0.002478,-0.022424,0.011607,0.002545,0.005087,0.022860,0.054522,-0.012804,-0.020959
std,71.193401,0.144374,1.005561,0.944504,1.004430,0.962282,0.998579,1.221294,1.022374,1.460470,...,1.004071,0.956890,1.117386,1.011196,1.011000,1.055255,0.965682,1.990361,1.102446,0.941150
min,0.000000,0.000000,-0.651134,-0.195586,-0.839721,-4.334982,-0.350082,-0.077211,-1.157572,-0.175191,...,-1.216654,-0.258763,-0.775069,-0.268728,-0.687065,-0.316404,-3.028056,-0.145223,-0.481220,-1.078546
25%,69.000000,1.000000,-0.651134,-0.195586,-0.839721,-0.618301,-0.350082,-0.077211,-0.688373,-0.175191,...,-1.216654,-0.258763,-0.605475,-0.268728,-0.687065,-0.316404,-0.592535,-0.145223,-0.425860,-0.586902
50%,118.000000,1.000000,-0.651134,-0.195586,-0.072787,-0.313843,-0.350082,-0.077211,-0.208747,-0.175191,...,0.821926,-0.258763,-0.416186,-0.268728,-0.438588,-0.316404,0.250562,-0.145223,-0.292992,-0.229622
75%,174.000000,1.000000,0.386341,-0.195586,0.694148,0.132445,-0.350082,-0.077211,0.498031,-0.175191,...,0.821926,-0.258763,0.194455,-0.268728,0.234649,-0.316404,0.762577,-0.145223,0.017012,0.289954
max,481.000000,1.000000,13.599292,25.603964,18.333643,21.316441,11.779527,48.270645,7.162266,73.993591,...,0.821926,21.113789,40.735954,9.343297,11.337988,32.767643,3.153850,115.020592,30.925158,12.043319


New Train/Test data splitting (50/50) for no interaction features Ro5/bRo5

In [17]:
# classifier/ → MELTING_POINT_2026/
PROJECT_ROOT = Path.cwd().parent        # directory above a path: .../MELTING_POINT_2026

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data_curation" / "processed_data"

data_path = processed_dir / "final_dataset_no_interaction.parquet"
df = pd.read_parquet(data_path)

print("Loaded:", data_path)
print("Shape:", df.shape)

df.head()


Loaded: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/data_curation/processed_data/final_dataset_no_interaction.parquet
Shape: (17220, 105)


,SMILES,MP,Type,Ro5,RDKit_SMR_VSA2,RDKit_Kappa2,MACCS_41,RDKit_SMR_VSA6,RDKit_SlogP_VSA5,RDKit_VSA_EState7,...,RDKit_VSA_EState9,RDKit_fr_Ar_NH,RDKit_Chi4n,RDKit_PEOE_VSA12,RDKit_NumAliphaticRings,MACCS_101,RDKit_TPSA,RDKit_fr_ArN,RDKit_fr_SH,RDKit_NumHeteroatoms
0,ON=Cc1cscc1,122.0,Train,1,0.0,2.470712,0,0.0,5.563451,1.399306,...,0.0,0,0.415115,0.0,0,0,32.59,0,0,3
1,O=C1CC[C@]2(C(=C1)CC[C@@H]1[C@@H]2[C@H](O)C[C@...,205.5,Train,1,0.0,4.636401,0,0.0,59.296141,6.544950,...,0.0,0,7.745421,0.0,5,1,66.90,0,0,4
2,[O-][n+]1ccccc1,64.0,Train,1,0.0,1.598931,0,0.0,0.000000,2.888889,...,0.0,0,0.382875,0.0,0,0,26.94,0,0,2
3,OC1CCC2(C(C1)CC=C1C2CCC2(C1CCC2C(CCC(C(C)C)C)C...,146.0,Train,1,0.0,7.994467,0,0.0,105.750639,15.793583,...,0.0,0,8.525564,0.0,4,1,20.23,0,0,1
4,CC(=O)c1ccc(cc1)Br,51.0,Train,1,0.0,2.971126,0,0.0,17.281726,0.000000,...,0.0,0,0.684597,0.0,0,0,17.07,0,0,2


In [18]:
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd

# 0) Start from your dataset
data_with_features = df.copy()  # or whatever your dataframe variable is

# 1) Remove current Type column (if it exists)
data_with_features = data_with_features.drop(columns=["Type"], errors="ignore")

# 2) 50/50 stratified split based on Ro5
split = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for train_index, test_index in split.split(data_with_features, data_with_features["Ro5"]):
    strat_train_set = data_with_features.iloc[train_index].copy()
    strat_test_set  = data_with_features.iloc[test_index].copy()

# 3) Add new Type labels
strat_train_set["Type"] = "Train"
strat_test_set["Type"]  = "Test"

# 4) Combine back together
final_data = pd.concat([strat_train_set, strat_test_set], axis=0).reset_index(drop=True)

# -----------------------------
# Checks
# -----------------------------
print(f"Train set: {final_data[final_data['Type'] == 'Train'].shape}")
print(f"Test set : {final_data[final_data['Type'] == 'Test'].shape}")

# Ro5 distribution check (works for 0/1 or True/False)
print("\nRo5 distribution (Train):")
print(final_data[final_data["Type"] == "Train"]["Ro5"].value_counts(normalize=True))

print("\nRo5 distribution (Test):")
print(final_data[final_data["Type"] == "Test"]["Ro5"].value_counts(normalize=True))

# If you specifically want "violation percentage" where violation == Ro5 == 0:
train_violation_pct = 100 * (final_data[final_data["Type"] == "Train"]["Ro5"].astype(int) == 0).mean()
test_violation_pct  = 100 * (final_data[final_data["Type"] == "Test"]["Ro5"].astype(int) == 0).mean()

print(f"\nTrain set: {train_violation_pct:.2f}% violate Ro5 (Ro5==0)")
print(f"Test set : {test_violation_pct:.2f}% violate Ro5 (Ro5==0)")


Train set: (8610, 105)
Test set : (8610, 105)

Ro5 distribution (Train):
Ro5
1    0.978746
0    0.021254
Name: proportion, dtype: float64

Ro5 distribution (Test):
Ro5
1    0.97863
0    0.02137
Name: proportion, dtype: float64

Train set: 2.13% violate Ro5 (Ro5==0)
Test set : 2.14% violate Ro5 (Ro5==0)


In [21]:
from pathlib import Path

# Choose where to save
out_path = Path.cwd() / "artifacts" / "full_data_50_50_no_interaction.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)

# Save as parquet
final_data.to_parquet(out_path, index=False)

print(f"✅ Saved stratified 50/50 dataset to: {out_path}")

✅ Saved stratified 50/50 dataset to: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/artifacts/full_data_50_50_no_interaction.parquet


In [22]:
# classifier/ → MELTING_POINT_2026/
PROJECT_ROOT = Path.cwd().parent        # directory above a path: .../MELTING_POINT_2026

df_path = PROJECT_ROOT / "Ro5" / "artifacts" / "full_data_50_50_no_interaction.parquet"
df = pd.read_parquet(df_path)

df.head()


,SMILES,MP,Ro5,RDKit_SMR_VSA2,RDKit_Kappa2,MACCS_41,RDKit_SMR_VSA6,RDKit_SlogP_VSA5,RDKit_VSA_EState7,RDKit_FpDensityMorgan2,...,RDKit_fr_Ar_NH,RDKit_Chi4n,RDKit_PEOE_VSA12,RDKit_NumAliphaticRings,MACCS_101,RDKit_TPSA,RDKit_fr_ArN,RDKit_fr_SH,RDKit_NumHeteroatoms,Type
0,CC(Oc1cc(N)c(cc1Cl)Cl)C,42.0,1,0.000000,4.072612,0,5.733667,13.847474,0.067663,1.846154,...,0,0.889091,0.000000,0,0,35.25,1,0,4,Train
1,Nc1ccc(cc1)S(=O)(=O)Nc1nnc(s1)C(C)(C)C,222.0,1,0.000000,5.023728,0,10.455762,25.778835,0.000000,1.750000,...,0,1.388409,5.131558,0,0,97.97,1,0,8,Train
2,O=S1(=O)CCCCO1,13.5,1,0.000000,2.027699,0,12.359736,12.841643,1.626551,2.000000,...,0,0.625719,0.000000,1,0,43.37,0,0,4,Train
3,Cc1cc(C)c2c(c1)CC2=O,48.0,1,0.000000,1.939976,0,0.000000,27.048343,0.650648,2.000000,...,0,1.748670,0.000000,1,1,17.07,0,0,1,Train
4,C#CC1(CCCCC1)OC(=N)O,96.0,1,5.409284,3.484918,0,0.000000,32.104108,9.201156,2.000000,...,0,1.583809,0.000000,1,0,53.31,0,0,3,Train


In [23]:
NON_FEATURES = ["SMILES", "MP", "Type", "Ro5"]
feature_cols = [c for c in df.columns if c not in NON_FEATURES]

# keep numeric only (important)
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

print("Number of feature columns:", len(feature_cols))


# 3) Fit scaler on TRAIN only
df_train = df[df["Type"] == "Train"]

scaler = StandardScaler()
scaler.fit(df_train[feature_cols])

# 4) Save scaler
scaler_path = PROJECT_ROOT / "Ro5" / "artifacts" / "df_final_scaler_50_50_no_interaction.pkl"
joblib.dump(scaler, scaler_path)

print("✅ Scaler fitted on TRAIN only and saved to:")
print(scaler_path)

# 5) Apply scaler to FULL dataset
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.transform(df_scaled[feature_cols])

print("Scaled dataframe shape:", df_scaled.shape)

# 6) Save scaled dataset
scaled_path = PROJECT_ROOT / "Ro5" / "artifacts" / "full_data_50_50_no_interaction_scaled.parquet"
df_scaled.to_parquet(scaled_path, index=False)

print("✅ Scaled dataset saved to:")
print(scaled_path)

Number of feature columns: 101
✅ Scaler fitted on TRAIN only and saved to:
/Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/artifacts/df_final_scaler_50_50_no_interaction.pkl
Scaled dataframe shape: (17220, 105)
✅ Scaled dataset saved to:
/Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/artifacts/full_data_50_50_no_interaction_scaled.parquet
